# **Python for Business and Marketing**

<div style="background-color: #fff3cd; padding: 10px; border-left: 5px solid #ffc107; border-radius: 4px;">
# Problem 1: 
Renaming, filtering and sorting the transport data
</div>

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
data = pd.read_csv("transport_data.csv")

# 1a) Set up column mapping to make headers clean and readable
column_map = {
    "trip_id": "ID",
    "origin": "Start",
    "destination": "End",
    "distance_km": "Distance (km)",
    "travel_time_min": "Time (min)",
    "ticket_price": "Price (£)",
    "day": "Day",
    "passenger_type": "Passenger",
    "delay_min": "Delay (min)"
}

def rename_columns(df, mapping):
    # Just passing the rename dictionary through
    return df.rename(columns=mapping)

data = rename_columns(data, column_map)


# 1b) Quick helper to filter rows based on a single column rule
def filter_rows(df, column, condition):
    # Evaluates the condition series and keeps only the rows where it's True
    return df[condition(df[column])]


# 1c) Multi-column filter using isin() and numpy to combine masks
def filter_rows_multi(df, conditions):
    masks = []
    for col, vals in conditions.items():
        if not isinstance(vals, (list, tuple, set)):
            vals = [vals]
        masks.append(df[col].isin(vals))
    
    # Combine all masks together so only rows matching ALL conditions pass
    combined_mask = np.all([*masks], axis=0)
    return df[combined_mask]


# 1d) Chain filtering and sorting together
def filter_and_sort(df, conditions, sort_by=None, ascending=True):
    result = filter_rows_multi(df, conditions)
    if sort_by is not None:
        result = result.sort_values(by=sort_by, ascending=ascending)
    return result


# Test Case
print("New column headers:", list(data.columns))

# Check how many journeys were delayed (delay > 0)
delayed = filter_rows(data, "Delay (min)", lambda x: x > 0)
print(f"\nOut of {len(data)} total trips, {len(delayed)} had a delay.")
display(delayed.head())

print("\nChecking journeys over 15 km:")
display(filter_rows(data, "Distance (km)", lambda x: x > 15).head())

# Filter for senior citizens traveling on weekends
weekend_seniors = filter_rows_multi(data, {"Day": ["Sat", "Sun"], "Passenger": ["senior"]})
print(f"\nFound {len(weekend_seniors)} senior trips on Sat or Sun:")
display(weekend_seniors.head())

print("\nSame weekend senior trips sorted by cheapest price first:")
display(filter_and_sort(data, {"Day": ["Sat", "Sun"], "Passenger": ["senior"]}, sort_by="Price (£)").head())

print("\nSorted by price and day, most expensive first:")
display(filter_and_sort(data, {"Day": ["Sat", "Sun"], "Passenger": ["senior"]}, sort_by=["Price (£)", "Day"], ascending=False).head())

New column headers: ['ID', 'Start', 'End', 'Distance (km)', 'Time (min)', 'Price (£)', 'Day', 'Passenger', 'Delay (min)']

Out of 600 total trips, 593 had a delay.


,ID,Start,End,Distance (km),Time (min),Price (£),Day,Passenger,Delay (min)
0,1,B,F,11.57,10.8,5.55,Sun,senior,3
1,2,E,D,10.69,12.8,6.44,Sat,adult,2
2,3,A,E,19.65,10.8,8.23,Sat,student,3
3,4,B,E,5.28,5.3,2.21,Tue,student,5
4,5,A,C,10.05,13.7,6.03,Sun,adult,7



Checking journeys over 15 km:


,ID,Start,End,Distance (km),Time (min),Price (£),Day,Passenger,Delay (min)
2,3,A,E,19.65,10.8,8.23,Sat,student,3
11,12,E,F,15.08,22.2,6.33,Thu,student,4
13,14,E,F,15.08,18.7,6.33,Mon,student,6
26,27,D,F,18.99,16.6,11.34,Sat,adult,3
29,30,D,F,18.99,18.1,11.34,Sat,adult,4



Found 28 senior trips on Sat or Sun:


,ID,Start,End,Distance (km),Time (min),Price (£),Day,Passenger,Delay (min)
0,1,B,F,11.57,10.8,5.55,Sun,senior,3
42,43,B,C,9.89,5.5,4.47,Sat,senior,1
54,55,A,B,11.05,11.2,5.34,Sun,senior,3
56,57,F,E,14.03,20.4,6.34,Sun,senior,9
71,72,C,E,8.51,9.8,4.85,Sat,senior,4



Same weekend senior trips sorted by cheapest price first:


,ID,Start,End,Distance (km),Time (min),Price (£),Day,Passenger,Delay (min)
478,479,B,E,5.28,5.2,2.53,Sat,senior,4
396,397,C,F,7.29,6.4,3.49,Sat,senior,5
424,425,F,C,8.17,37.0,3.92,Sat,senior,4
198,199,F,C,8.17,6.8,3.92,Sun,senior,1
42,43,B,C,9.89,5.5,4.47,Sat,senior,1



Sorted by price and day, most expensive first:


,ID,Start,End,Distance (km),Time (min),Price (£),Day,Passenger,Delay (min)
541,542,A,E,19.65,19.7,9.43,Sun,senior,4
465,466,D,F,18.99,21.3,9.12,Sat,senior,1
238,239,E,C,17.59,17.9,8.44,Sat,senior,4
432,433,E,C,17.59,15.6,8.44,Sat,senior,2
416,417,E,F,15.08,24.3,7.28,Sun,senior,5


<div style="background-color: #fff3cd; padding: 10px; border-left: 5px solid #ffc107; border-radius: 4px;">
# Problem 2: 
Analysing the journeys
</div>

In [2]:
# 2a) Count how many trips happen on a specific day from a specific start point
def count_journeys(df, day, start):
    subset = filter_rows_multi(df, {"Day": [day], "Start": [start]})
    return len(subset)


# 2b) Find which start station has the shortest vs longest average trip distance
def mean_distance_by_start(df):
    means = df.groupby("Start")["Distance (km)"].mean()
    min_loc = means.idxmin()
    max_loc = means.idxmax()

    return {
        "min": {"location": min_loc, "value": round(float(means[min_loc]), 2)},
        "max": {"location": max_loc, "value": round(float(means[max_loc]), 2)}
    }


# 2c) Calculate average price per km depending on who is travelling
def mean_price_per_km(df, passenger_type):
    # Price divided by distance for every single row
    price_per_km = df["Price (£)"] / df["Distance (km)"]
    means = price_per_km.groupby(df["Passenger"]).mean()

    if passenger_type not in means.index:
        print(f"Note: No data available for passenger type '{passenger_type}'.")
        return None

    return round(float(means[passenger_type]), 4)


# 2d) Find best-running and worst-delayed route pairs (Start-End)
def delay_extremes(df):
    avg_delay = df.groupby(["Start", "End"])["Delay (min)"].mean()
    least = avg_delay.idxmin()
    most = avg_delay.idxmax()

    return {
        "least_delayed": {"journey": f"{least[0]}-{least[1]}", "avg_delay": round(float(avg_delay[least]), 2)},
        "most_delayed": {"journey": f"{most[0]}-{most[1]}", "avg_delay": round(float(avg_delay[most]), 2)}
    }


# Test Case
print("Journeys on Monday leaving from A:", count_journeys(data, "Mon", "A"))
print("Journeys on Sunday leaving from F:", count_journeys(data, "Sun", "F"))

print("\nAverage distance extremes by start location:")
print(mean_distance_by_start(data))

print("\nAverage price per km by passenger category:")
for p in ["adult", "student", "senior"]:
    print(f"  {p}: £{mean_price_per_km(data, p)}")

# Testing missing category
mean_price_per_km(data, "child")

print("\nDelay extremes across routes:")
print(delay_extremes(data))

Journeys on Monday leaving from A: 17
Journeys on Sunday leaving from F: 27

Average distance extremes by start location:
{'min': {'location': 'B', 'value': 9.44}, 'max': {'location': 'A', 'value': 13.55}}

Average price per km by passenger category:
  adult: £0.6071
  student: £0.4241
  senior: £0.4806
Note: No data available for passenger type 'child'.

Delay extremes across routes:
{'least_delayed': {'journey': 'A-B', 'avg_delay': 3.26}, 'most_delayed': {'journey': 'A-C', 'avg_delay': 7.16}}


<div style="background-color: #fff3cd; padding: 10px; border-left: 5px solid #ffc107; border-radius: 4px;">
# Problem 3: 
Conditional probabilities
</div>

In [3]:
# 3a) Find the probability distribution of End locations given a Start and Passenger type
def most_likely_end(df, start, passenger):
    given = filter_rows_multi(df, {"Start": [start], "Passenger": [passenger]})
    if len(given) == 0:
        print("No records found for this combination.")
        return None

    probs = {}
    for end in sorted(df["End"].unique()):
        # Intersection count divided by condition count
        joint = filter_rows_multi(df, {"End": [end], "Start": [start], "Passenger": [passenger]})
        probs[end] = len(joint) / len(given)

    print(f"P(End | Start = {start}, Passenger = {passenger}):")
    for end, p in probs.items():
        print(f"   {end}: {p:.3f}")

    # Return the destination with the highest probability
    return max(probs, key=probs.get)


# 3b) General function for conditional probability with any two given conditions
def most_likely_category(df, target_col, col1, val1, col2, val2):
    given = filter_rows_multi(df, {col1: [val1], col2: [val2]})
    if len(given) == 0:
        print(f"No records matching {col1}={val1} and {col2}={val2}.")
        return None

    probs = {}
    for cat in sorted(df[target_col].unique()):
        joint = filter_rows_multi(df, {target_col: [cat], col1: [val1], col2: [val2]})
        probs[cat] = len(joint) / len(given)

    print(f"P({target_col} | {col1} = {val1}, {col2} = {val2}):")
    for cat, p in probs.items():
        print(f"   {cat}: {p:.3f}")

    return max(probs, key=probs.get)


# Test Case
print("Most likely End:", most_likely_end(data, "A", "senior"), "\n")
print("Most likely End:", most_likely_end(data, "C", "student"), "\n")

print("Most likely Passenger type for A -> B:", 
      most_likely_category(data, "Passenger", "Start", "A", "End", "B"), "\n")

print("Most likely Start location for students on Saturdays:", 
      most_likely_category(data, "Start", "Day", "Sat", "Passenger", "student"), "\n")

print("Checking generic function matches 3a result:")
print("Most likely End:", 
      most_likely_category(data, "End", "Start", "A", "Passenger", "senior"))

P(End | Start = A, Passenger = senior):
   A: 0.000
   B: 0.450
   C: 0.150
   D: 0.100
   E: 0.200
   F: 0.100
Most likely End: B 

P(End | Start = C, Passenger = student):
   A: 0.214
   B: 0.286
   C: 0.000
   D: 0.143
   E: 0.167
   F: 0.190
Most likely End: B 

P(Passenger | Start = A, End = B):
   adult: 0.348
   senior: 0.391
   student: 0.261
Most likely Passenger type for A -> B: senior 

P(Start | Day = Sat, Passenger = student):
   A: 0.312
   B: 0.062
   C: 0.125
   D: 0.062
   E: 0.188
   F: 0.250
Most likely Start location for students on Saturdays: A 

Checking generic function matches 3a result:
P(End | Start = A, Passenger = senior):
   A: 0.000
   B: 0.450
   C: 0.150
   D: 0.100
   E: 0.200
   F: 0.100
Most likely End: B


<div style="background-color: #fff3cd; padding: 10px; border-left: 5px solid #ffc107; border-radius: 4px;">
# Problem 4: 
Ticket discounts for students and seniors
</div>

In [4]:
# 4a) Calculate discount percentage compared to adult baseline price for a specific route
def route_discounts(df, start, end):
    discounts = {"student_discount": None, "senior_discount": None}
    route = filter_rows_multi(df, {"Start": [start], "End": [end]})

    # Adults are the baseline full price
    adult_prices = route.loc[route["Passenger"] == "adult", "Price (£)"]
    if len(adult_prices) == 0:
        return discounts

    adult_price = adult_prices.mean()

    for ptype, key in [("student", "student_discount"), ("senior", "senior_discount")]:
        prices = route.loc[route["Passenger"] == ptype, "Price (£)"]
        if len(prices) > 0:
            disc = (adult_price - prices.mean()) / adult_price * 100
            discounts[key] = round(float(disc), 2)

    return discounts


# 4b) Loop through all routes to find where students and seniors get the biggest price cuts
def best_discounts(df):
    locations = sorted(df["Start"].unique())
    best = {
        "students": {"discount": None, "start": None, "end": None},
        "seniors": {"discount": None, "start": None, "end": None}
    }

    for s in locations:
        for e in locations:
            if s == e:
                continue  # skip traveling to the same place
            
            d = route_discounts(df, s, e)
            for group, key in [("students", "student_discount"), ("seniors", "senior_discount")]:
                val = d[key]
                if val is not None:
                    if best[group]["discount"] is None or val > best[group]["discount"]:
                        best[group] = {"discount": val, "start": s, "end": e}

    return best


# 4c) Build the price map dictionary containing standard and sunday rates for students and seniors
def create_price_map(df):
    price_map = {}
    locations = sorted(df["Start"].unique())

    for start in locations:
        price_map[start] = {}
        for end in locations:
            if start == end:
                continue

            adults = filter_rows_multi(df, {"Start": [start], "End": [end], "Passenger": ["adult"]})
            if len(adults) == 0:
                continue

            adult_price = round(float(adults["Price (£)"].mean()), 2)

            # Applying fixed discount rules:
            # - Students: 20% off normally, 25% off on Sundays
            # - Seniors: 10% off normally, 15% off on Sundays
            price_map[start][end] = {
                "adult_price": adult_price,
                "student_price": round(adult_price * 0.80, 2),
                "student_sun_price": round(adult_price * 0.75, 2),
                "senior_price": round(adult_price * 0.90, 2),
                "senior_sun_price": round(adult_price * 0.85, 2)
            }

    return price_map


# Test Case
print("Discounts for A to B:", route_discounts(data, "A", "B"))
print("Discounts for E to C:", route_discounts(data, "E", "C"))
print("Discounts for A to A (should be None):", route_discounts(data, "A", "A"))

print("\nRoutes with the biggest discounts:")
print(best_discounts(data))

# Generate and check the price map structure
price_map = create_price_map(data)
print("\nSample price mapping for route A -> B:")
print(price_map["A"]["B"])

Discounts for A to B: {'student_discount': 30.02, 'senior_discount': 19.46}
Discounts for E to C: {'student_discount': 25.31, 'senior_discount': 20.0}
Discounts for A to A (should be None): {'student_discount': None, 'senior_discount': None}

Routes with the biggest discounts:
{'students': {'discount': 46.96, 'start': 'D', 'end': 'A'}, 'seniors': {'discount': 40.54, 'start': 'D', 'end': 'A'}}

Sample price mapping for route A -> B:
{'adult_price': 6.63, 'student_price': 5.3, 'student_sun_price': 4.97, 'senior_price': 5.97, 'senior_sun_price': 5.64}
